# 06. v3 — 하이브리드 검색 + 리랭커

v2까지는 **벡터 검색 하나**로 상위 5개를 골랐습니다. v3는 검색을 두 단계로 바꿉니다. (구조도 3)
1. **1차 검색:** 벡터 검색(뜻이 비슷한 청크) + **BM25**(단어가 일치하는 청크)를 각각 20개씩 찾고, **RRF**로 순위를 합쳐 후보 20개
2. **리랭킹:** 후보 20개를 **리랭커**(BAAI/bge-reranker-v2-m3)가 질문과 한 쌍씩 다시 읽고 채점 → 상위 5개
3. **관련도 확인:** 리랭커 1등 점수가 기준(`RELEVANCE_THRESHOLD`)보다 낮으면 LLM을 부르지 않고 "모름"

- 검색 코드: `app/retrieval/` (서버와 같은 코드. 노트북은 벡터 검색만 pgvector 대신 v2의 FAISS 인덱스를 씀)
- **v2와 같게 두는 것:** 청크(v2 인덱스 그대로), 답변 프롬프트, 답변 모델, 근거 5개, 채점기, 테스트셋
- **API 사용:** ④ 질문 임베딩(약 $0.001), ⑦ 평가(약 $0.09). ④의 리랭커는 내 컴퓨터에서 실행(무료, CPU로 10분 이상 걸림)

## ① 설정

In [1]:
import pickle
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*langchain-community.*")

import numpy as np
import pandas as pd
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from app.config import settings
from app.generation.answer import ANSWER_PROMPT, format_context
from app.retrieval.hybrid import HybridRetriever
from app.retrieval.rerank import Reranker
from eval.rag_eval import load_testset, recall_hit, run_eval, summarize, JudgeResult

TOP_K, CANDIDATES = 5, 20
RESULTS_DIR = PROJECT_DIR / "eval" / "results"
CACHE_PATH = settings.data_dir / "index" / "v3_candidates.pkl"
print("리랭커:", settings.reranker_model, "| 후보 수:", CANDIDATES, "| 최종 근거 수:", TOP_K)

리랭커: BAAI/bge-reranker-v2-m3 | 후보 수: 20 | 최종 근거 수: 5


## ② v2 청크·인덱스 불러오기
v2 노트북에서 만든 `data/index/v2_faiss`를 그대로 씁니다. (청크가 같아야 "검색 방법"만 비교할 수 있음)

In [2]:
embeddings = OpenAIEmbeddings(model=settings.embedding_model)
vectorstore = FAISS.load_local(str(settings.data_dir / "index" / "v2_faiss"), embeddings,
                               allow_dangerous_deserialization=True)   # 직접 만든 파일
docs = [vectorstore.docstore.search(vectorstore.index_to_docstore_id[i]) for i in range(vectorstore.index.ntotal)]
chunks = [{"id": i, "document_id": d.metadata["doc_id"], "doc_code": d.metadata["doc_id"],
           "page": d.metadata["page"], "kind": d.metadata["kind"], "content": d.page_content}
          for i, d in enumerate(docs)]
print("청크 수:", len(chunks), pd.Series([c["kind"] for c in chunks]).value_counts().to_dict())

청크 수: 610 {'table': 357, 'text': 152, 'chart': 101}


## ③ 검색기 만들기
- 벡터 검색: 질문을 임베딩해 FAISS에서 가까운 청크 번호를 가져옴
- BM25: 형태소 분석기(kiwipiepy)로 청크를 단어 조각으로 잘라 색인 (예: "청년층의" → "청년층")

In [3]:
def faiss_search(query, k, document_ids=None):
    q = np.array([embeddings.embed_query(query)], dtype="float32")
    _, idx = vectorstore.index.search(q, k)
    return [int(i) for i in idx[0] if i >= 0]

reranker = Reranker(settings.reranker_model)
retriever = HybridRetriever(chunks, faiss_search, reranker)

from app.retrieval.bm25 import tokenize
print("BM25 단어 조각 예:", tokenize("2026년 5월 청년층의 경제활동참가율은?"))

BM25 단어 조각 예: ['2026', '년', '5', '월', '청년', '층', '경제', '활동', '참가', '율']


## ④ 110문항 후보 검색 + 리랭킹 (처음 한 번만, 결과 저장)
문항마다 후보 20개를 찾고 리랭커 점수를 붙여 `data/index/v3_candidates.pkl`에 저장합니다.
여기서 계산한 순위로 ⑤의 여러 방식을 **추가 비용 없이** 비교합니다.

In [4]:
testset = load_testset(PROJECT_DIR / "eval" / "testset.csv")
if CACHE_PATH.exists():
    candidates = pickle.loads(CACHE_PATH.read_bytes())
    print("불러옴:", CACHE_PATH)
else:
    candidates = {}
    for n, row in enumerate(testset.itertuples(), start=1):
        hits = retriever.retrieve(row.question, top_k=CANDIDATES, candidates=CANDIDATES)
        candidates[row.id] = [h.__dict__ for h in hits]
        if n % 10 == 0:
            print(f"{n}/{len(testset)} 완료")
    CACHE_PATH.write_bytes(pickle.dumps(candidates))
    print("저장:", CACHE_PATH)

/home/user/git_project/project6/ReportQA/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:  34%|███▍      | 135/393 [00:00<00:00, 1339.59it/s]

Loading weights:  70%|██████▉   | 274/393 [00:00<00:00, 1363.55it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1823.05it/s]

10/110 완료


... (6 outputs left out)

80/110 완료


90/110 완료


100/110 완료


110/110 완료
저장: /home/user/git_project/project6/ReportQA/data/index/v3_candidates.pkl


## ⑤ 검색 방식별 Recall@5 비교 (무료)
같은 후보 목록에서 순서만 바꿔 상위 5개를 뽑습니다.
- **벡터만** = v2 방식 · **BM25만** · **하이브리드** = RRF 순위 · **하이브리드 + 리랭커** = v3

In [5]:
def top5(hits, key):
    ranked = [h for h in hits if key(h) is not None]
    ranked.sort(key=key)
    return ranked[:TOP_K]

METHODS = {
    "벡터만 (v2)": lambda h: h["vector_rank"],
    "BM25만": lambda h: h["bm25_rank"],
    "하이브리드 (RRF)": lambda h: -h["rrf_score"],
    "하이브리드 + 리랭커 (v3)": lambda h: -h["rerank_score"],
}
answerable = testset[testset["type"] != "unknown"]
rows = []
for name, key in METHODS.items():
    hit = {}
    for row in answerable.itertuples():
        found = [(chunks[h["index"]]["doc_code"], chunks[h["index"]]["page"]) for h in top5(candidates[row.id], key)]
        hit[row.id] = recall_hit(testset.set_index("id").loc[row.id], found)
    s = pd.Series(hit)
    rows.append({"방식": name, "Recall@5": s.mean(),
                 **{t: s[answerable.set_index("id")["type"] == t].mean() for t in ["text", "table", "chart"]}})
recall_table = pd.DataFrame(rows).set_index("방식").round(3)
recall_table

,Recall@5,text,table,chart
방식,,,,
벡터만 (v2),0.92,1.000,0.900,0.833
BM25만,0.96,1.000,1.000,0.867
하이브리드 (RRF),0.99,1.000,1.000,0.967
하이브리드 + 리랭커 (v3),0.98,0.975,0.967,1.000


## ⑥ "모름" 기준 점수 정하기
리랭커 1등 점수를 "문서에 답이 있는 문항"과 "모름 문항"으로 나눠 봅니다.
기준을 높이면 모름 문항을 더 잘 거르지만, 답이 있는 문항까지 "모름"으로 답할 위험이 커집니다.
→ **답이 있는 문항을 하나도 거르지 않는 가장 높은 기준**을 고릅니다.

In [6]:
top_score = pd.Series({qid: max(h["rerank_score"] for h in hs) for qid, hs in candidates.items()})
is_unknown_q = testset.set_index("id")["type"] == "unknown"
print("모름 문항 1등 점수:", top_score[is_unknown_q].round(3).sort_values().tolist())
print("답이 있는 문항 1등 점수 (낮은 5개):", top_score[~is_unknown_q].round(3).sort_values().head(5).tolist())

grid = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3]
th_table = pd.DataFrame([{"기준": t,
                          "모름 문항 거름": int((top_score[is_unknown_q] < t).sum()),
                          "답 있는 문항 잘못 거름": int((top_score[~is_unknown_q] < t).sum())} for t in grid])
THRESHOLD = max([t for t in grid if (top_score[~is_unknown_q] < t).sum() == 0])
print("선택한 기준:", THRESHOLD)
th_table

모름 문항 1등 점수: [0.117, 0.222, 0.754, 0.898, 0.953, 0.974, 0.989, 0.992, 0.993, 0.996]
답이 있는 문항 1등 점수 (낮은 5개): [0.5, 0.86, 0.94, 0.957, 0.966]
선택한 기준: 0.3


,기준,모름 문항 거름,답 있는 문항 잘못 거름
0,0.000,0,0
1,0.001,0,0
2,0.005,0,0
3,0.010,0,0
4,0.020,0,0
5,0.050,0,0
6,0.100,0,0
7,0.200,1,0
8,0.300,2,0


## ⑦ v3 평가 — API 사용 (약 $0.09)
리랭커 상위 5개로 답하고(프롬프트는 v0~v2와 같음), 1등 점수가 기준보다 낮으면 LLM 없이 "모름"으로 답합니다.

In [7]:
llm = ChatOpenAI(model=settings.llm_model)
qid_by_question = dict(zip(testset["question"], testset["id"]))

def answer_v3(question):
    hits = top5(candidates[qid_by_question[question]], METHODS["하이브리드 + 리랭커 (v3)"])
    used = [chunks[h["index"]] for h in hits]
    retrieved = [(c["doc_code"], c["page"]) for c in used]
    if not hits or hits[0]["rerank_score"] < THRESHOLD:
        return "모름", retrieved
    return llm.invoke(ANSWER_PROMPT.format(context=format_context(used), question=question)).text, retrieved

RESULT_PATH = RESULTS_DIR / "v3.csv"
if RESULT_PATH.exists():
    results = pd.read_csv(RESULT_PATH)
    print("불러옴:", RESULT_PATH)
else:
    judge_llm = ChatOpenAI(model=settings.judge_model).with_structured_output(JudgeResult)
    with get_usage_metadata_callback() as usage:
        results = run_eval(testset, answer_v3, judge_llm, k=TOP_K)
    results.to_csv(RESULT_PATH, index=False)
    print("저장:", RESULT_PATH)
    for model, u in usage.usage_metadata.items():
        print(f"{model}: 입력 {u['input_tokens']:,} / 출력 {u['output_tokens']:,} 토큰")

10/110 완료


20/110 완료


30/110 완료


40/110 완료


50/110 완료


60/110 완료


70/110 완료


80/110 완료


90/110 완료


100/110 완료


110/110 완료
저장: /home/user/git_project/project6/ReportQA/eval/results/v3.csv
gpt-6-luna: 입력 228,252 / 출력 5,876 토큰
gpt-6.1-terra: 입력 32,380 / 출력 3,706 토큰


## ⑧ v0 ~ v3 비교

In [8]:
versions = {v: pd.read_csv(RESULTS_DIR / f"{v}.csv") for v in ["v0", "v1", "v2"]}
versions["v3"] = results
compare = pd.DataFrame({v: summarize(r) for v, r in versions.items()})
compare["v2→v3(%p)"] = ((compare["v3"] - compare["v2"]) * 100).round(1)
print("판정 분포 v3:", results["judge"].value_counts().to_dict())
compare

판정 분포 v3: {'correct': 105, 'wrong': 3, 'refused': 2}


,v0,v1,v2,v3,v2→v3(%p)
Recall@5,0.870,0.890,0.920,0.980,6.0
정답률,0.673,0.736,0.882,0.955,7.3
텍스트,0.925,0.950,0.950,0.925,-2.5
표,0.533,0.833,0.900,0.967,6.7
차트(vector),0.611,0.444,0.833,0.944,11.1
차트(image),0.000,0.000,0.583,1.000,41.7
환각률,0.000,0.000,0.000,0.000,0.0


In [9]:
pd.set_option("display.max_colwidth", 70)
both = (versions["v2"][["id", "type", "judge", "recall_hit"]]
        .merge(results[["id", "judge", "recall_hit", "answer_text"]], on="id", suffixes=("_v2", "_v3"))
        .merge(testset[["id", "question", "answer"]], on="id"))
improved = both[(both["judge_v2"] != "correct") & (both["judge_v3"] == "correct")]
regressed = both[(both["judge_v2"] == "correct") & (both["judge_v3"] != "correct")]
print(f"좋아진 문항 {len(improved)}개 / 나빠진 문항 {len(regressed)}개")
display(improved[["id", "type", "question", "recall_hit_v2", "recall_hit_v3"]])
display(regressed[["id", "type", "question", "answer", "answer_text", "recall_hit_v2", "recall_hit_v3"]])

좋아진 문항 11개 / 나빠진 문항 3개


,id,type,question,recall_hit_v2,recall_hit_v3
2,T003,text,2026년 5월 휴학경험자의 평균 휴학기간은?,True,True
13,B001,table,2026년 5월 15세 이상 전체 실업률은?,False,True
16,B004,table,2026년 5월 청년층 졸업 취업자 중 정보통신업 취업자 수는?,True,True
23,C001,chart,2022년 5월 15~19세 인구의 전년동월대비 증감은?,False,True
24,C002,chart,2023년 5월 15~19세 인구의 전년동월대비 증감은?,False,True
25,C003,chart,2023년 5월 20~24세 인구의 전년동월대비 증감은?,False,True
27,C005,chart,2025년 5월 15~19세 인구의 전년동월대비 증감은?,False,True
31,C009,chart,2025년 5월 20~24세 최종학교 졸업자의 전년동월대비 증감은?,False,True
61,B023,table,2024년 4분기 콘텐츠산업 상장사 매출액은?,False,True
86,C017,chart,2019년 60세 이상 고령자 중 현재 자녀와 따로 사는 비중은?,True,True


,id,type,question,answer,answer_text,recall_hit_v2,recall_hit_v3
5,T006,text,2026년 5월 최종학교 졸업 후 취업경험자 비율은?,84.8%,모름,True,False
46,T025,text,2025년 4분기 실태조사의 조사 기간은?,2026년 1월 14일 ~ 2월 13일,2026년 4월 14일부터 5월 13일까지입니다. [D02 p.43],True,True
77,B026,table,2025년 여자의 노후 준비 방법 중 예금·적금 비율은?,20.7%,모름,True,True


## ⑨ 발견한 것

### 1. 점수 변화
| 지표 | v0 | v1 | v2 | **v3** | v2→v3 |
|---|---|---|---|---|---|
| 정답률 | 67.3% | 73.6% | 88.2% | **95.5%** | +7.3%p |
| Recall@5 | 87.0% | 89.0% | 92.0% | **98.0%** | +6.0%p |
| 텍스트 | 92.5% | 95.0% | 95.0% | 92.5% | -2.5%p |
| 표 | 53.3% | 83.3% | 90.0% | **96.7%** | +6.7%p |
| 차트(벡터) | 61.1% | 44.4% | 83.3% | **94.4%** | +11.1%p |
| 차트(이미지) | 0% | 0% | 58.3% | **100%** | +41.7%p |
| 환각률 | 0% | 0% | 0% | 0% | 0 |

**v2에서 남은 오답의 대부분이 "검색 실패"였는데, v3에서 해결됐습니다.** 좋아진 11문항 중 7문항이 v2에서 정답 쪽을 찾지 못했던 문항입니다.

### 2. 검색 방식별 Recall@5 (⑤, 같은 후보에서 순서만 바꿔 비교)
| 방식 | 전체 | 텍스트 | 표 | 차트 |
|---|---|---|---|---|
| 벡터만 (v2) | 92% | 100% | 90.0% | 83.3% |
| BM25만 | 96% | 100% | 100% | 86.7% |
| 하이브리드 (RRF) | **99%** | 100% | 100% | 96.7% |
| 하이브리드 + 리랭커 (v3) | 98% | 97.5% | 96.7% | **100%** |

- **BM25 하나만으로도 벡터 검색보다 높았습니다.** 통계 질문은 "2025년 4분기", "15~19세", "음악"처럼 **정확한 단어**가 중요해서, 뜻이 비슷한 것을 찾는 벡터 검색보다 단어 일치가 유리합니다.
- **두 방식을 합치면(RRF) 99%.** 한쪽이 놓친 것을 다른 쪽이 찾습니다.
- **리랭커를 더하면 Recall@5는 오히려 1%p 낮았습니다** (하이브리드 99% → 98%). 정답률은 비용을 아끼려고 v3(하이브리드 + 리랭커)만 측정했기 때문에, **리랭커 단독 효과는 확인하지 못했습니다.** 다음 실험 후보: 하이브리드만 쓴 정답률 측정(약 $0.08).
  참고: C017(2019년 자녀와 비동거 70.7%), C026(2015년 소득 부족 64.7%)은 v2에서도 정답 쪽을 찾았지만 다른 연도·계열 값을 골랐고, v3에서는 정답.
- 그림 차트 C001·C002·C003·C005·C009: 캡션은 `'22.5`(22년 5월) 형식인데 질문은 "2022년 5월"이라 v2(벡터 검색)는 정답 쪽을 5위 안에 넣지 못했음 → v3에서 모두 찾아 **차트(이미지) 12문항 모두 정답**

### 3. "모름" 기준 점수 (⑥)
- 답이 있는 100문항의 리랭커 1등 점수: 가장 낮은 것이 **0.5**
- 모름 10문항: 0.117, 0.222, 0.754 … 0.996 → 대부분 높음 (문서 안에 "비슷한 주제"의 글이 있기 때문)
- **기준 0.3 선택:** 답이 있는 문항은 하나도 거르지 않으면서 모름 문항 2개를 LLM 호출 없이 거름 (`RELEVANCE_THRESHOLD=0.3`)
- 나머지 모름 8문항은 답변 프롬프트의 "근거에 없으면 모름" 규칙이 걸러서 **환각률 0% 유지** (구조도 3의 "모름 두 번 거르기"가 실제로 둘 다 쓰임)
- 주의: 같은 110문항으로 기준을 정하고 평가했으므로 기준이 이 테스트셋에 맞춰져 있을 수 있습니다. 문서를 늘리면 다시 확인이 필요합니다.

### 4. 나빠진 3문항
| 문항 | 원인 |
|---|---|
| T006 (취업경험자 비율 84.8%) | 리랭커가 정답 쪽(7·17쪽)을 5위 밖으로 밀어냄 (Recall 실패) |
| T025 (조사 기간) | D02의 다른 분기 조사 기간(4월 14일~5월 13일)을 고름. 비슷한 문장이 여러 쪽에 있음 |
| B026 (여자 예금·적금 20.7%) | 정답 쪽은 찾았지만 "모름" |

### 5. 비용·속도
- 리랭커: 내 컴퓨터 CPU에서 실행 (API 비용 0). 노트북 전체 실행 약 38분, 대부분이 110문항 × 후보 20개 리랭킹 (문항당 십수 초, 청크가 최대 800자로 길고 다른 작업과 CPU를 나눠 씀)
- 평가 API 비용: 약 $0.08 (답변 + 채점)
- 서버에서 질문 1개를 처리할 때도 리랭킹 때문에 몇 초가 더 걸립니다. 느리면 `RETRIEVE_TOP_K`를 20 → 10으로 줄일 수 있습니다. (계획서 13장 리스크 대응)